In [ ]:
!pip install tensorflow
!pip install pillow
!pip install matplotlib
!pip install opencv-python
!pip install tifffile
!pip install numpy
!pip install scikit-image
!pip install graphviz
!pip install pydot

In [3]:
# Import the required libraries
import tensorflow as tf
from tensorflow import keras
from keras import layers, models, optimizers
from keras.layers import Input, LeakyReLU, Reshape, Dropout, Dense, Flatten, BatchNormalization, Activation, ZeroPadding2D, Conv2D, Conv2DTranspose
from keras.models import Sequential, Model
from keras.optimizers import Adam, RMSprop
import numpy as np
from skimage.transform import rotate
from keras.utils import plot_model
from keras.callbacks import TensorBoard

# Set up a callback to log the model architecture to TensorBoard
log_dir = 'logs/gan'
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

import matplotlib.pyplot as plt
import cv2
import tifffile
import math
from PIL import Image, ImageDraw


#if tf.test.gpu_device_name(): 
#    print('Default GPU Device: {}'.format(tf.test.gpu_device_name()))
#else:
#    print("Please install GPU version of TF")

# create a method to build a generator model that generates
# images of the same size as the input images
# considering 256x256x5 as the input image size
# without using dense layers. instead use conv2dtranspose layers to
# save memory and increase speed.
def build_generator():
    inputs = Input((100,))
    
    x = inputs

    x = layers.Dense(8 * 8 * 1024)(x)
    x = layers.LeakyReLU()(x)
    x = layers.Reshape((8, 8, 1024))(x)

    #x = layers.Conv2DTranspose(512, 4, strides=1, padding='same')(x)
    #x = layers.BatchNormalization()(x)
    #x = layers.ReLU()(x)

    #x = layers.Conv2DTranspose(256, 4, strides=2, padding='same')(x)
    #x = layers.BatchNormalization()(x)
    #x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(128, 4, strides=4, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(64, 4, strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    outputs = layers.Conv2DTranspose(3, 4, strides=2, padding='same', activation='tanh')(x)
    
    # Load pre-trained VGG16 model
    vgg16_model = tf.keras.applications.vgg16.VGG16(weights='imagenet', include_top=False, input_shape=(128,128,3))

    input_shape = (128,128,3)
    new_input = Input(shape=input_shape)
    x = Conv2D(3, (1, 1), padding='same')(new_input)

    # Connect the modified input layer to the rest of the VGG16 model
    x = vgg16_model(x)

    # Freeze VGG16 layers
    for layer in vgg16_model.layers:
        layer.trainable = False

    # Create feature extraction model using VGG16
    vgg16_feature_extractor = tf.keras.Sequential()
    for layer in vgg16_model.layers[:-4]:
        vgg16_feature_extractor.add(layer)

    # Define the final model
    generator_model = tf.keras.Model(inputs, outputs, name='generator')

    # Define the final model
    generator_output = outputs[:, :, :, :3]
    generator_output = vgg16_feature_extractor(generator_output)
    generator_output = tf.keras.Model(inputs, generator_output, name='generator_vgg16')
    generator_output.summary()

    return generator_model


# based on the previous method, create a method to build the discriminator model
"""
def build_discriminator():
    inputs = Input((128, 128, 3))
    
    x = inputs
    x = layers.Conv2D(64, 4, strides=2, padding='same')(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Conv2D(128, 4, strides=2, padding='same')(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Conv2D(256, 4, strides=2, padding='same')(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Conv2D(512, 4, strides=2, padding='same')(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Flatten()(x)
    outputs = x
    model = Model(inputs, outputs)
    model.summary()

    return Model(inputs, outputs)
"""
def build_discriminator():

    vgg16_model = tf.keras.applications.vgg16.VGG16(weights='imagenet', include_top=False, input_shape=(128,128,3))

    # Set VGG16 layers as non-trainable
    for layer in vgg16_model.layers:
        layer.trainable = False

    # Define input layer
    input_shape = (128,128,3)
    inputs = Input(shape=input_shape)

    # Extract features from input image using VGG16
    features = vgg16_model(inputs)

    # Add flatten and dense layers for classification
    flatten = layers.Flatten()(features)
    dense1 = layers.Dense(256, activation='relu')(flatten)
    dense2 = layers.Dense(1, activation='sigmoid')(dense1)

    # Define the final model
    discriminator_model = tf.keras.Model(inputs, dense2, name='discriminator')

    # Compile the model
    discriminator_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

    discriminator_model.summary()

    return discriminator_model

generator = build_generator()
discriminator = build_discriminator()

# Define the GAN model as a combination of generator and discriminator models
discriminator.trainable = True
gan_input = keras.Input(shape=(100,))
gan_output = discriminator(generator(gan_input))
gan = keras.Model(gan_input, gan_output, name="gan")
plot_model(gan, to_file='gan.png', show_shapes=True)

# Compile the models
generator_optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9)
discriminator_optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9)
discriminator.compile(loss="binary_crossentropy", optimizer=discriminator_optimizer)
gan.compile(loss="binary_crossentropy", optimizer=generator_optimizer)

plot_model(generator, to_file='generator.png', show_shapes=True, show_layer_names=True)
plot_model(discriminator, to_file='discriminator.png', show_shapes=True, show_layer_names=True)
plot_model(gan, to_file='gan.png', show_shapes=True, show_layer_names=True)

# Load the .tif images and preprocess them
import os

# Define the root directory where the images are located
root_dir = "./"

# Recursively find all the .tif files in the root directory and its subdirectories
image_paths = []
for dirpath, _, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.endswith("_UNHEALTHY.tif"):
            image_path = os.path.join(dirpath, filename)
            image_paths.append(image_path)
            
images = []
print()
for path in image_paths:
    # Load the TIFF image
    image = tifffile.imread(path)

    # Display the image shape
    # print(image.shape)

    # image = cv2.imread(path, cv2.IMREAD_UNCHANGED)

    # Convert the image to a NumPy array and normalize the pixel values to the range [-1, 1]
    # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.astype("float32") / 127.5 - 1.0
    print(image.shape)
    # Selecting just the RGB colors for while
    image = image[:, :, :3]
    print(image.shape)
    # Resize the image to the desired size using OpenCV and append the preprocessed image to the list of images
    image = cv2.resize(image, (128, 128), interpolation=cv2.INTER_LINEAR)
    
    images.append(image)
    

    # Rotate the image
    rotated_image = rotate(image, angle=180, order=1, mode='reflect', preserve_range=True)
    images.append(rotated_image)

print(len(images))
train_images = np.array(images)
train_dataset = tf.data.Dataset.from_tensor_slices(train_images)
train_dataset = train_dataset.shuffle(buffer_size=32).batch(8)

# Define a function to generate images using the trained GAN model
def generate_images(model, noise, epoch):
    # Generate images from noise using the generator model
    generated_images = model.predict(noise)
    #print(generated_images.shape)
    # Rescale pixel values to the range [0, 1]
    generated_images = (1/(2*2.25)) * generated_images + 0.5

    # Save the generated images
    for i in range(generated_images.shape[0]):
        #print(generated_images[i])
        for j in range(generated_images.shape[-1]):
            tifffile.imwrite(f"generated_images/{epoch}_{i}_{j}.tif", generated_images[i][:,:,j], dtype='float32')
        #tifffile.imwrite(f"generated_images/{epoch}_00002_{i}.tif", [generated_images[i][3],generated_images[i][2],generated_images[i][1]], shape=[256, 256])
        
# Train the GAN model on the preprocessed dataset
epochs = 10000
noise_dim = 100
num_examples_to_generate = 10
seed = tf.random.normal([num_examples_to_generate, noise_dim])

for epoch in range(epochs):
    print(f"Epoch {epoch+1}")

    for real_images in train_dataset:
        # Generate random noise
        #noise = tf.random.normal([real_images.shape[0], noise_dim, 5])
        noise = np.random.normal(0, 1, (10, 100))
        noise = noise.reshape(10, 100)

        # Generate fake images using the generator model
        fake_images = generator.predict(noise)

        # Concatenate real and fake images
        combined_images = tf.concat([real_images, fake_images], axis=0)
        # Create labels for real and fake images
        real_labels = tf.ones((real_images.shape[0], 1))
        fake_labels = tf.zeros((fake_images.shape[0], 1))
        combined_labels = tf.concat([real_labels, fake_labels], axis=0)

        # Train the discriminator model
        discriminator_loss = discriminator.train_on_batch(combined_images, combined_labels)

        # Train the generator model
        noise = tf.random.normal([real_images.shape[0], noise_dim])
        generator_loss = gan.train_on_batch(noise, real_labels)
        print(generator_loss)


    # Generate images using the trained GAN model
    if epoch % 10 == 0:
        generate_images(generator, seed, epoch)

# Save the generator model
generator.save("generator_model.h5")

Model: "generator_vgg16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_13 (InputLayer)       [(None, 100)]             0         
                                                                 
 dense_6 (Dense)             (None, 65536)             6619136   
                                                                 
 leaky_re_lu_2 (LeakyReLU)   (None, 65536)             0         
                                                                 
 reshape_2 (Reshape)         (None, 8, 8, 1024)        0         
                                                                 
 conv2d_transpose_6 (Conv2DT  (None, 32, 32, 128)      2097280   
 ranspose)                                                       
                                                                 
 batch_normalization_4 (Batc  (None, 32, 32, 128)      512       
 hNormalization)                                   

KeyboardInterrupt: 